# Accelerating Monte Carlo FFE Resampling

This notebook benchmarks four implementations of the same Monte Carlo
simulation for First Floor Elevation (FFE) resampling across ~450 k NJ buildings.
Each version receives **exactly the same input** (`nsi_clean`, a GeoDataFrame)
and drops the geometry column internally before the numerical work,
so the comparisons are apples-to-apples.

| Stage | Approach | Implementation |
|-------|----------|----------------|
| 1  | CPU loop          | Outer loop over 2,000 simulations + inner loop over 6 foundation types |
| 2  | CPU vectorised    | Outer loop over 6 foundation types only - one big NumPy call each |
| 3a | GPU initial       | Same vectorised logic on GPU; full cuDF dataframe per chunk, fixed 500 k-row chunks |
| 3b | GPU optimized     | Fused sampler, int-only transfer, VRAM-aware chunking with pool cleanup |

Implementation 3a runs out of GPU memory when the GPU is already partly in use;
3b applies memory optimizations that fix this while producing identical results.

### Setup

In [1]:
import os
import json
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import fiona

import cudf
import cupy as cp

print(f'cudf  {cudf.__version__}  |  cupy  {cp.__version__}')
print(f'GPU:  {cp.cuda.runtime.getDeviceProperties(0)["name"].decode()}')

cudf  26.04.00  |  cupy  14.1.0
GPU:  NVIDIA A100-PCIE-40GB


### Load data

In [2]:
path = "/scratch/gpfs/GVILLARI/hackathon_2026/FULL_NJ/nsi_new_jersey.gpkg"

print('layers:', fiona.listlayers(path))
nsi = gpd.read_file(path, layer='nsi')

print(f'shape:   {nsi.shape}')
print(f'crs:     {nsi.crs}')
print(f'columns: {list(nsi.columns)}')
print(f'\nnull counts:\n{nsi.isnull().sum()}')

layers: ['nsi']
shape:   (2766189, 30)
crs:     EPSG:4326
columns: ['fd_id', 'bid', 'cbfips', 'st_damcat', 'occtype', 'bldgtype', 'num_story', 'sqft', 'found_type', 'found_ht', 'med_yr_blt', 'val_struct', 'val_cont', 'val_vehic', 'ftprntid', 'ftprntsrc', 'source', 'students', 'pop2amu65', 'pop2amo65', 'pop2pmu65', 'pop2pmo65', 'o65disable', 'u65disable', 'x', 'y', 'firmzone', 'grnd_elv_m', 'ground_elv', 'geometry']

null counts:
fd_id               0
bid                 0
cbfips              0
st_damcat           0
occtype             0
bldgtype            0
num_story           0
sqft                0
found_type          0
found_ht            0
med_yr_blt          0
val_struct          0
val_cont            0
val_vehic           0
ftprntid            0
ftprntsrc      403451
source              0
students            0
pop2amu65           0
pop2amo65           0
pop2pmu65           0
pop2pmo65           0
o65disable          0
u65disable          0
x                   0
y                

In [3]:
# Setting output directory
OUT_PATH = "/scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs"
os.makedirs(OUT_PATH, exist_ok=True)
print(f'Saving outputs to: {OUT_PATH}')

Saving outputs to: /scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs


### Column selection & cleaning

We keep only the columns needed for the simulation. The `geometry` column
stays in `nsi_clean` so it can be reattached to outputs at the end,
but **every simulation function drops it before any numerical work**.

In [4]:
COLS_NSI = [
    'fd_id', 'cbfips', 'st_damcat', 'occtype', 'bldgtype',
    'num_story', 'found_type', 'found_ht', 'grnd_elv_m', 'ground_elv',
    'geometry', 'val_struct', 'val_cont',
]

nsi_clean = nsi[COLS_NSI].copy()

# Strip sub-type suffix from occtype (e.g. RES1-2SNB → RES1)
nsi_clean['occtype'] = nsi_clean['occtype'].str.replace(r'-.*$', '', regex=True)

print(f'shape after column drop: {nsi_clean.shape}')
print(f'foundation types:        {sorted(nsi_clean.found_type.unique())}')
nsi_clean.head()

shape after column drop: (2766189, 13)
foundation types:        ['B', 'C', 'I', 'P', 'S', 'W']


,fd_id,cbfips,st_damcat,occtype,bldgtype,num_story,found_type,found_ht,grnd_elv_m,ground_elv,geometry,val_struct,val_cont
0,548509648,340297280001017,RES,RES1,W,2.0,C,1.5,1.961297,6.434703,POINT (-74.08485 39.91152),356900.839,178450.419
1,548511068,340297131001010,RES,RES1,W,1.0,B,2.0,2.652716,8.703138,POINT (-74.10459 40.1006),293271.182,146635.591
2,548523256,340410311013002,RES,RES1,W,1.0,B,2.0,101.671425,333.567678,POINT (-75.03149 40.96023),423172.098,211586.049
3,548545953,340297310021035,RES,RES1,W,2.0,B,2.0,2.471919,8.109971,POINT (-74.14944 39.89421),384874.181,192437.090
4,548549026,340297140004002,RES,RES1,M,1.0,B,2.0,13.874615,45.520391,POINT (-74.15328 40.04653),285159.837,142579.918


### Triangular distribution parameters

First Floor Elevation offsets are modelled with triangular distributions.
Parameters `(low, mode, high)` per foundation type come from the UNSAFE / FATHOM framework.

In [5]:
FFE_PARAMS = {
    'B': (0.0, 0.5, 1.5),    # Basement
    'C': (0.0, 1.5, 4.0),    # Crawl space
    'S': (0.0, 1.5, 4.0),    # Slab
    'I': (6.0, 9.0, 12.0),   # Pile
    'W': (6.0, 9.0, 12.0),   # Solid Wall
    'P': (6.0, 9.0, 12.0),   # Pier
}

### Implementation 1 - CPU loop (baseline)

**Structure:** outer loop over 2,000 simulations × inner loop over 6 foundation types
= 12,000 small calls to `rng.triangular`.

Each call pays full Python function-call overhead and returns a tiny array.


In [6]:
def resample_ffe(df, found_type_col='found_type', n_sims=2000, seed=None):
    """
    Baseline CPU Monte Carlo: outer loop = simulations, inner loop = foundation types.
    Geometry is dropped before computation and not returned - caller reattaches.
    """
    # Drop geometry so all three implementations receive the same tabular input
    if hasattr(df, 'geometry'):
        df = pd.DataFrame(df.drop(columns='geometry'))

    rng = np.random.default_rng(seed)
    df  = df.copy()
    n   = len(df)

    ffe_sims = np.empty((n, n_sims))

    for i in range(n_sims):                          # 2 000 iterations
        ffes = np.zeros(n)
        for fnd_type in FFE_PARAMS:                  # 6 iterations
            mask = df[found_type_col] == fnd_type
            if mask.any():
                left, mode, right = FFE_PARAMS[fnd_type]
                ffes[mask] = rng.triangular(left, mode, right, size=mask.sum())
        ffe_sims[:, i] = ffes

    df['ffe_q05'] = np.round(np.percentile(ffe_sims,  5, axis=1), 2)
    df['ffe_q25'] = np.round(np.percentile(ffe_sims, 25, axis=1), 2)
    df['ffe_q50'] = np.round(np.percentile(ffe_sims, 50, axis=1), 2)
    df['ffe_q75'] = np.round(np.percentile(ffe_sims, 75, axis=1), 2)
    df['ffe_q95'] = np.round(np.percentile(ffe_sims, 95, axis=1), 2)

    return df


start    = time.time()
nsi_cpu  = resample_ffe(nsi_clean, found_type_col='found_type', n_sims=2000, seed=42)
cpu_time = time.time() - start

print(f'CPU loop time: {cpu_time:.1f}s  ({len(nsi_cpu):,} rows)')
nsi_cpu[['found_type', 'ffe_q05', 'ffe_q25', 'ffe_q50', 'ffe_q75', 'ffe_q95']].head(10)

CPU loop time: 2311.8s  (2,766,189 rows)


,found_type,ffe_q05,ffe_q25,ffe_q50,ffe_q75,ffe_q95
0,C,0.54,1.18,1.78,2.47,3.33
1,B,0.20,0.45,0.64,0.89,1.22
2,B,0.19,0.45,0.65,0.90,1.22
3,B,0.20,0.44,0.65,0.89,1.23
4,B,0.20,0.44,0.64,0.88,1.22
5,C,0.56,1.22,1.77,2.44,3.29
6,S,0.55,1.24,1.77,2.47,3.27
7,B,0.19,0.43,0.63,0.89,1.22
8,B,0.20,0.44,0.65,0.90,1.22
9,I,6.91,8.09,9.00,9.89,11.05


### Implementation 2 - CPU vectorised

**Key insight:** flip the loop order

NumPy's `rng.triangular(size=(count, n_sims))` generates the entire
`(count × 2000)` block in a single C-level call -
eliminating 12,000 Python-level round trips.

In [7]:
def resample_ffe_vectorized(df, found_type_col='found_type', n_sims=2000, seed=None):
    """
    Vectorised CPU Monte Carlo: one (count × n_sims) block per foundation type.
    Geometry is dropped before computation and not returned - caller reattaches.
    """
    # Drop geometry - same as CPU loop and GPU versions
    if hasattr(df, 'geometry'):
        df = pd.DataFrame(df.drop(columns='geometry'))

    rng  = np.random.default_rng(seed)
    df   = df.copy()
    n    = len(df)
    found = df[found_type_col].to_numpy()

    ffe_sims = np.zeros((n, n_sims), dtype=np.float32)

    for fnd_type, (left, mode, right) in FFE_PARAMS.items():   # 6 iterations only
        mask  = found == fnd_type
        count = int(mask.sum())
        if count:
            ffe_sims[mask, :] = rng.triangular(left, mode, right,
                                               size=(count, n_sims))

    qs = np.percentile(ffe_sims, [5, 25, 50, 75, 95], axis=1)
    for col, q in zip(['ffe_q05', 'ffe_q25', 'ffe_q50', 'ffe_q75', 'ffe_q95'], qs):
        df[col] = np.round(q, 2)

    return df


start    = time.time()
nsi_vec  = resample_ffe_vectorized(nsi_clean, found_type_col='found_type', n_sims=2000, seed=42)
vec_time = time.time() - start

print(f'CPU loop:   {cpu_time:.1f}s')
print(f'Vectorised: {vec_time:.1f}s')
print(f'Speedup:    {cpu_time / vec_time:.1f}x')
nsi_vec[['found_type', 'ffe_q05', 'ffe_q25', 'ffe_q50', 'ffe_q75', 'ffe_q95']].head(10)

CPU loop:   2311.8s
Vectorised: 227.2s
Speedup:    10.2x


,found_type,ffe_q05,ffe_q25,ffe_q50,ffe_q75,ffe_q95
0,C,0.55,1.25,1.76,2.37,3.25
1,B,0.20,0.43,0.63,0.90,1.23
2,B,0.19,0.43,0.63,0.88,1.23
3,B,0.20,0.43,0.62,0.87,1.18
4,B,0.18,0.44,0.64,0.88,1.24
5,C,0.58,1.22,1.77,2.41,3.25
6,S,0.56,1.22,1.79,2.41,3.27
7,B,0.19,0.43,0.65,0.88,1.24
8,B,0.20,0.43,0.64,0.89,1.23
9,I,6.97,8.15,8.98,9.85,11.09


### Implementation 3

### a GPU (cuDF + CuPy), initial version

The vectorised approach maps directly onto GPU parallelism:
each foundation-type block becomes thousands of simultaneous threads.

**Two extra considerations vs the CPU version:**

1. **Inverse-CDF triangular sampler** - CuPy has no built-in `triangular`,
   so we draw `U ~ Uniform(0, 1)` and apply the closed-form inverse CDF.

2. **Chunked processing** - the full `(n_buildings × n_sims)` float32 matrix
   is several GB, so rows are processed in fixed 500 k-row chunks to keep
   GPU VRAM under control.

Each chunk is moved to the GPU as a full cuDF dataframe, the quantiles are
computed, and the result is brought back with `.to_pandas()`.


In [ ]:
# # do we need the df = df.copy() and also why do we need to chunk_size? 
# def resample_ffe_gpu(df, found_type_col='found_type', n_sims=2000, seed=None, chunk_size=500_000):
#     if hasattr(df, 'geometry'):
#         df = pd.DataFrame(df.drop(columns='geometry'))

#     fnd_types = list(FFE_PARAMS.keys())
#     fnd_to_int = {f: i for i, f in enumerate(fnd_types)}
#     df = df.copy()
#     df[found_type_col] = df[found_type_col].map(fnd_to_int)

#     if seed is not None:
#         cp.random.seed(seed)

#     n = len(df)
#     results = []

#     for start_idx in range(0, n, chunk_size):
#         end_idx = min(start_idx + chunk_size, n)
#         chunk = df.iloc[start_idx:end_idx]
#         gdf_chunk = cudf.from_pandas(chunk)

#         nc = len(gdf_chunk)
#         ffe_sims = cp.zeros((nc, n_sims), dtype=cp.float32)
#         found = gdf_chunk[found_type_col].to_cupy()

#         for i, (fnd_type, (left, mode, right)) in enumerate(FFE_PARAMS.items()):
#             mask  = found == i
#             count = int(mask.sum())
#             if count:
#                 ffe_sims[mask, :] = _triangular_cupy(left, mode, right,
#                                                      size=(count, n_sims))

#         qs = cp.round(
#             cp.percentile(ffe_sims, [5, 25, 50, 75, 95], axis=1).T, 2,
#         )

#         for j, col in enumerate(['ffe_q05', 'ffe_q25', 'ffe_q50', 'ffe_q75', 'ffe_q95']):
#             gdf_chunk[col] = qs[:, j]

#         results.append(gdf_chunk.to_pandas())
#         print(f"processed {end_idx:,} / {n:,}")

#     return pd.concat(results, ignore_index=True)


# start = time.time()
# nsi_gpu = resample_ffe_gpu(nsi_clean, found_type_col='found_type', n_sims=2000, seed=42)
# gpu_time = time.time() - start

# print(f"GPU time:      {gpu_time:.1f}s")
# print(f"CPU time:      {cpu_time:.1f}s")
# print(f"speedup:       {cpu_time / gpu_time:.1f}x")
# nsi_gpu[['ffe_q05', 'ffe_q25', 'ffe_q50', 'ffe_q75', 'ffe_q95']].head(10)

### b GPU optimized

The initial version ran out of GPU memory: the sampler allocated a separate
full-size array for every arithmetic step, the entire dataframe was copied to
the GPU when only one column was needed, and memory was never freed between
chunks. The statistics are unchanged - these are memory optimizations only.

**Three changes:**

1. **Fused sampler** - the inverse-CDF transform is wrapped in `@cp.fuse()`,
   so it runs as a *single* GPU kernel with no large intermediate arrays.
   Random draws are also generated as float32 directly (no float64 → float32
   copy).

2. **Auto-sized chunks + memory cleanup** - chunk size is computed at runtime
   from the actual free VRAM (`cp.cuda.runtime.memGetInfo()`) instead of a
   fixed 500 k, and the CuPy memory pool is freed after every chunk so usage
   does not accumulate.

3. **Minimal host↔device transfer** - only the integer `found_type` column is
   moved to the GPU; string columns stay on the host and only the five
   quantile columns are copied back, avoiding the full cuDF round trip.

In [19]:
@cp.fuse()
def _tri_transform(u, left, right, span_low, span_high, fc):
    """Fused inverse-CDF triangular transform — one kernel, no temp arrays."""
    return cp.where(
        u < fc,
        left  + cp.sqrt(u         * span_low),
        right - cp.sqrt((1.0 - u) * span_high),
    )

In [20]:
def resample_ffe_gpu(df, found_type_col='found_type', n_sims=2000,
                     seed=None, chunk_size=None, mem_fraction=0.6):
    """GPU Monte Carlo FFE resampling. Only the integer found_type column is
    moved to the GPU; quantiles come back to host per chunk."""
    if hasattr(df, 'geometry'):
        df = pd.DataFrame(df.drop(columns='geometry'))

    fnd_to_int = {f: i for i, f in enumerate(FFE_PARAMS.keys())}
    int_to_fnd = {i: f for f, i in fnd_to_int.items()}

    df = df.copy()
    found_all = df[found_type_col].map(fnd_to_int).to_numpy()   # host int array
    n = len(df)

    if seed is not None:
        cp.random.seed(seed)

    # Auto-size chunks from actual free VRAM. Peak ≈ chunk*n_sims*4 * ~3.
    if chunk_size is None:
        free_b, total_b = cp.cuda.runtime.memGetInfo()
        bytes_per_row = n_sims * 4 * 3
        chunk_size = max(10_000, int(free_b * mem_fraction / bytes_per_row))
        print(f'free VRAM {free_b/1e9:.1f} / {total_b/1e9:.1f} GB '
              f'→ chunk_size {chunk_size:,}')

    qcols = ['ffe_q05', 'ffe_q25', 'ffe_q50', 'ffe_q75', 'ffe_q95']
    out_q = np.empty((n, 5), dtype=np.float32)
    pool  = cp.get_default_memory_pool()

    for start_idx in range(0, n, chunk_size):
        end_idx = min(start_idx + chunk_size, n)
        nc      = end_idx - start_idx
        found   = cp.asarray(found_all[start_idx:end_idx])      # int column only

        ffe_sims = cp.zeros((nc, n_sims), dtype=cp.float32)
        for i, (left, mode, right) in enumerate(FFE_PARAMS.values()):
            mask  = found == i
            count = int(mask.sum())
            if count:
                u = cp.random.uniform(0.0, 1.0, size=(count, n_sims),
                                      dtype=cp.float32)            # float32 directly
                ffe_sims[mask, :] = _tri_transform(
                    u,
                    cp.float32(left), cp.float32(right),
                    cp.float32((right - left) * (mode  - left)),   # span_low
                    cp.float32((right - left) * (right - mode)),   # span_high
                    cp.float32((mode  - left) / (right - left)),   # fc
                )
                del u

        q = cp.percentile(ffe_sims, [5, 25, 50, 75, 95], axis=1).T   # (nc, 5)
        out_q[start_idx:end_idx] = cp.asnumpy(cp.round(q, 2))

        del ffe_sims, found, q
        pool.free_all_blocks()
        print(f'  processed {end_idx:,} / {n:,}')

    for j, col in enumerate(qcols):
        df[col] = out_q[:, j]
    df[found_type_col] = pd.Series(found_all).map(int_to_fnd).values  # restore strings
    return df

In [21]:
start    = time.time()
nsi_gpu  = resample_ffe_gpu(nsi_clean, found_type_col='found_type', n_sims=2000, seed=42)
gpu_time = time.time() - start

print(f'\nGPU time:          {gpu_time:.1f}s')
print(f'CPU loop time:     {cpu_time:.1f}s')
print(f'Vectorised time:   {vec_time:.1f}s')
print(f'GPU vs CPU loop:   {cpu_time / gpu_time:.1f}x speedup')
print(f'GPU vs vectorised: {vec_time / gpu_time:.1f}x speedup')
nsi_gpu[['found_type', 'ffe_q05', 'ffe_q25', 'ffe_q50', 'ffe_q75', 'ffe_q95']].head(10)

free VRAM 17.2 / 42.4 GB → chunk_size 430,987
  processed 430,987 / 2,766,189
  processed 861,974 / 2,766,189
  processed 1,292,961 / 2,766,189
  processed 1,723,948 / 2,766,189
  processed 2,154,935 / 2,766,189
  processed 2,585,922 / 2,766,189
  processed 2,766,189 / 2,766,189

GPU time:          13.3s
CPU loop time:     2311.8s
Vectorised time:   227.2s
GPU vs CPU loop:   174.1x speedup
GPU vs vectorised: 17.1x speedup


,found_type,ffe_q05,ffe_q25,ffe_q50,ffe_q75,ffe_q95
0,C,0.54,1.20,1.74,2.38,3.26
1,B,0.18,0.43,0.64,0.89,1.23
2,B,0.20,0.45,0.63,0.87,1.21
3,B,0.20,0.43,0.62,0.87,1.21
4,B,0.18,0.43,0.62,0.86,1.22
5,C,0.59,1.22,1.75,2.39,3.28
6,S,0.55,1.21,1.74,2.45,3.31
7,B,0.19,0.43,0.64,0.88,1.25
8,B,0.19,0.43,0.63,0.87,1.21
9,I,6.95,8.13,9.02,9.91,11.09


### Save outputs

Geometry is reattached from `nsi_clean` before writing GeoPackage files.
Timings are also written to `timings.json` for easy reference.

In [23]:
ffe_cols = ['ffe_q05', 'ffe_q25', 'ffe_q50', 'ffe_q75', 'ffe_q95']

# ── CPU loop output ─────────────────────────────────────────────────
nsi_cpu_gpd = nsi_clean[['fd_id', 'geometry']].copy()
nsi_cpu_gpd[ffe_cols] = nsi_cpu[ffe_cols].values
nsi_cpu_gpd.to_file(f'{OUT_PATH}/nsi_ffe_cpu.gpkg', driver='GPKG')
print(f'saved: {OUT_PATH}/nsi_ffe_cpu.gpkg')

# ── Vectorised CPU output ───────────────────────────────────────────
nsi_vec_gpd = nsi_clean[['fd_id', 'geometry']].copy()
nsi_vec_gpd[ffe_cols] = nsi_vec[ffe_cols].values
nsi_vec_gpd.to_file(f'{OUT_PATH}/nsi_ffe_vec.gpkg', driver='GPKG')
print(f'saved: {OUT_PATH}/nsi_ffe_vec.gpkg')

# ── GPU output ──────────────────────────────────────────────────────
nsi_gpu_gpd = nsi_clean[['fd_id', 'geometry']].copy()
nsi_gpu_gpd[ffe_cols] = nsi_gpu[ffe_cols].values
nsi_gpu_gpd.to_file(f'{OUT_PATH}/nsi_ffe_gpu.gpkg', driver='GPKG')
print(f'saved: {OUT_PATH}/nsi_ffe_gpu.gpkg')

# ── Timings ─────────────────────────────────────────────────────────
timings = {
    'cpu_loop_s':          round(cpu_time, 2),
    'cpu_vectorised_s':    round(vec_time, 2),
    'gpu_s':               round(gpu_time, 2),
    'speedup_vec_vs_loop': round(cpu_time / vec_time, 1),
    'speedup_gpu_vs_loop': round(cpu_time / gpu_time, 1),
    'speedup_gpu_vs_vec':  round(vec_time / gpu_time, 1),
    'n_buildings':         len(nsi_clean),
    'n_sims':              2000,
}
with open(f'{OUT_PATH}/timings.json', 'w') as f:
    json.dump(timings, f, indent=2)
print(f'saved: {OUT_PATH}/timings.json')

saved: /scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs/nsi_ffe_cpu.gpkg
saved: /scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs/nsi_ffe_vec.gpkg
saved: /scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs/nsi_ffe_gpu.gpkg
saved: /scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs/timings.json


### Speedup summary

All three methods produce the same quantile columns from the same input.
The table below shows the relative performance.

In [25]:
print(f'  Buildings:              {len(nsi_clean):>10,}')
print(f'  Simulations per row:    {2000:>10,}')
print(f'  CPU loop:               {cpu_time:>9.1f}s')
print(f'  CPU vectorised:         {vec_time:>9.1f}s   {cpu_time/vec_time:>5.1f}x vs loop')
print(f'  GPU (cuDF + CuPy):      {gpu_time:>9.1f}s   {cpu_time/gpu_time:>5.1f}x vs loop  |  {vec_time/gpu_time:>4.1f}x vs vec')


  Buildings:               2,766,189
  Simulations per row:         2,000
  CPU loop:                  2311.8s
  CPU vectorised:             227.2s    10.2x vs loop
  GPU (cuDF + CuPy):           13.3s   174.1x vs loop  |  17.1x vs vec
